# kaggle-vllm 0.2.0 Qwen TP=2 sharded-state regression — recreated

Final focused regression for the **existing** `waqasm86/kaggle-vllm-models` artifact after public `kaggle-vllm==0.2.0` publication.

This notebook does **not** regenerate, reshard, modify, or re-upload model weights.

It is intentionally robust to a Kaggle Input / Hugging Face cache layout where the attached model directory contains symlinks. `kaggle_vllm.inspect_sharded_model()` correctly rejects symlink-backed model directories as a safety rule, so this notebook:

1. identifies the real attached Qwen artifact;
2. records its real rank/part filenames and file sizes;
3. runs direct strict inspection when the directory is symlink-free;
4. otherwise treats the strict symlink rejection as expected and builds a **tiny topology-only fixture** containing regular zero-byte shard placeholders plus real required metadata;
5. validates TP=2 topology and TP mismatch rejection against that fixture;
6. separately validates explicit symlink rejection;
7. loads and generates from the **real Qwen artifact** in a child process;
8. records machine-readable evidence.

Run only in a fresh Kaggle **GPU T4 x2** session with Internet enabled for SDK/bootstrap access. Prefer an attached Kaggle Input copy of the already-published Qwen artifact. Do not turn on the large Hub download unless necessary.


In [1]:
from pathlib import Path
import json, time

IDENTITY = "KAGGLE-VLLM-QWEN-REGRESSION-V3"
SENTINEL_DIR = Path("/kaggle/working/kaggle-vllm-020-qwen-regression")
SENTINEL_DIR.mkdir(parents=True, exist_ok=True)
SENTINEL = SENTINEL_DIR / "qwen-regression-started.json"
SENTINEL.write_text(json.dumps({
    "identity": IDENTITY,
    "started_unix": time.time(),
    "status": "STARTED"
}, indent=2) + "\n", encoding="utf-8")
print("=" * 72)
print(IDENTITY)
print("Sentinel created:", SENTINEL)
print("If you do NOT see this exact marker, the wrong notebook is running.")
print("=" * 72)


KAGGLE-VLLM-QWEN-REGRESSION-V3
Sentinel created: /kaggle/working/kaggle-vllm-020-qwen-regression/qwen-regression-started.json
If you do NOT see this exact marker, the wrong notebook is running.


In [2]:
from pathlib import Path
from importlib import metadata
import json
import os
import platform
import re
import shutil
import subprocess
import sys
import tempfile
import time

EXPECTED_SDK_VERSION = "0.2.0"
MODEL_REPO = "waqasm86/kaggle-vllm-models"

# Historical immutable model-content revision used by the project.
# README-only commits may exist later on main; do not confuse those with a
# regenerated model artifact.
MODEL_REVISION = "08bb62d0b68d20062e9009a9769c0df53d3dae21"

EXPECTED_NATIVE_WHEEL = (
    "vllm-0.18.2.dev0+ga26e8dc7f.d20260822.cu128-"
    "cp312-cp312-linux_x86_64.whl"
)
EXPECTED_NATIVE_SHA256 = (
    "5a9bd710b8a19fdd23abb3442baad892da977466f996334decd533a225f5fd0c"
)
EXPECTED_NATIVE_HF_REVISION = "f6b4f10de54924ed6fe9e28cceab84eca7276ab6"

EXPECTED_SHARDS = {
    f"model-rank-{rank}-part-{part}.safetensors"
    for rank in (0, 1)
    for part in (0, 1)
}

ALLOW_HUB_DOWNLOAD = False
WORK = Path("/kaggle/working/kaggle-vllm-020-qwen-regression")
RUNTIME = WORK / "runtime"
STAGED = RUNTIME / "vllm-staged"
OVERLAY = RUNTIME / "vllm-runtime-overlay"
MANIFEST = RUNTIME / "kaggle-vllm-runtime.json"
CACHE = Path("/kaggle/working/kaggle-vllm-cache")
EVIDENCE = WORK / "qwen-regression-evidence.json"
QWEN_LOG = WORK / "qwen-tp2-generation.log"
FIXTURE_ROOT = WORK / "inspection-fixture"

WORK.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version)
print("Platform:", platform.platform())
assert sys.version_info[:3] == (3, 12, 13), sys.version

subprocess.run(["nvidia-smi", "-L"], check=True)
subprocess.run(["nvidia-smi", "topo", "-m"], check=True)

import torch

torch_before = {
    "version": torch.__version__,
    "cuda": torch.version.cuda,
    "path": str(Path(torch.__file__).resolve()),
}
assert torch_before["version"] == "2.10.0+cu128", torch_before
assert torch_before["cuda"] == "12.8", torch_before
assert torch.cuda.device_count() == 2
assert all(torch.cuda.get_device_name(i) == "Tesla T4" for i in range(2))
assert all(torch.cuda.get_device_capability(i) == (7, 5) for i in range(2))

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        f"kaggle-vllm[hub]=={EXPECTED_SDK_VERSION}",
    ],
    check=True,
)

import kaggle_vllm

assert kaggle_vllm.__version__ == EXPECTED_SDK_VERSION
print("SDK:", metadata.version("kaggle-vllm"))
print("Torch before bootstrap:", torch_before)


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.12.90+-x86_64-with-glibc2.35
GPU 0: Tesla T4 (UUID: GPU-6ce138a9-c3e2-e50f-f151-65baccf00208)
GPU 1: Tesla T4 (UUID: GPU-c9563691-77ee-e8ba-cc2d-5889a3412754)
	GPU0	GPU1	CPU Affinity	NUMA Affinity	GPU NUMA ID
GPU0	 X 	PHB	0-3	0		N/A
GPU1	PHB	 X 	0-3	0		N/A

Legend:

  X    = Self
  SYS  = Connection traversing PCIe as well as the SMP interconnect between NUMA nodes (e.g., QPI/UPI)
  NODE = Connection traversing PCIe as well as the interconnect between PCIe Host Bridges within a NUMA node
  PHB  = Connection traversing PCIe as well as a PCIe Host Bridge (typically the CPU)
  PXB  = Connection traversing multiple PCIe bridges (without traversing the PCIe Host Bridge)
  PIX  = Connection traversing at most a single PCIe bridge
  NV#  = Connection traversing a bonded set of # NVLinks
SDK: 0.2.0
Torch before bootstrap: {'version': '2.10.0+cu128', 'cuda': '12.8', 'path': '/usr/local/lib/python3.12/dist-packages/torc

## Strict immutable runtime bootstrap and activation

This repeats only the runtime identity needed by the Qwen regression. The public-package acceptance notebook already performed the broader final acceptance matrix.


In [3]:
if RUNTIME.exists():
    shutil.rmtree(RUNTIME)

BOOTSTRAP = [
    "kaggle-vllm",
    "bootstrap",
    "--strict",
    "--staged", str(STAGED),
    "--overlay", str(OVERLAY),
    "--cache", str(CACHE),
    "--manifest", str(MANIFEST),
]

subprocess.run(BOOTSTRAP + ["--dry-run", "--json"], check=True)
subprocess.run(BOOTSTRAP, check=True)

manifest = json.loads(MANIFEST.read_text(encoding="utf-8"))
assert manifest["wheel"]["filename"] == EXPECTED_NATIVE_WHEEL
assert manifest["wheel"]["sha256"] == EXPECTED_NATIVE_SHA256
assert manifest["wheel"]["hf_revision"] == EXPECTED_NATIVE_HF_REVISION

from kaggle_vllm import activate_runtime

assert activate_runtime(MANIFEST)

import vllm
import vllm._C
import vllm._moe_C
import vllm.cumem_allocator

for module in (vllm, vllm._C, vllm._moe_C, vllm.cumem_allocator):
    assert Path(module.__file__).resolve().is_relative_to(STAGED.resolve()), module.__file__

doctor = subprocess.run(
    ["kaggle-vllm", "doctor", "--strict", "--json"],
    check=True,
    capture_output=True,
    text=True,
)
doctor_payload = json.loads(doctor.stdout)
assert doctor_payload["compatible"] is True

torch_after = {
    "version": torch.__version__,
    "cuda": torch.version.cuda,
    "path": str(Path(torch.__file__).resolve()),
}
assert torch_after == torch_before, (torch_before, torch_after)

print("Strict bootstrap, native imports, doctor, and Torch preservation: PASS")


{
  "profile": "kaggle-t4x2-cu128",
  "strict": true,
  "compatible": true,
  "findings": [
    {
      "check": "Python implementation",
      "status": "pass",
      "message": "Python implementation: CPython"
    },
    {
      "check": "Python ABI",
      "status": "pass",
      "message": "Python ABI: cp312"
    },
    {
      "check": "operating system",
      "status": "pass",
      "message": "operating system: Linux"
    },
    {
      "check": "machine",
      "status": "pass",
      "message": "machine: x86_64"
    },
    {
      "check": "Kaggle runtime",
      "status": "pass",
      "message": "Kaggle runtime: True"
    },
    {
      "check": "PyTorch",
      "status": "pass",
      "message": "PyTorch: 2.10.0+cu128"
    },
    {
      "check": "PyTorch CUDA",
      "status": "pass",
      "message": "PyTorch CUDA: 12.8"
    },
    {
      "check": "visible GPU count",
      "status": "pass",
      "message": "visible GPU count: 2"
    },
    {
      "check": "GPU model"

## Resolve the existing Qwen artifact safely

Selection rules:

- `KAGGLE_VLLM_QWEN_PATH` wins when explicitly provided.
- Otherwise search only `/kaggle/input` for the four known TP=2 shard filenames.
- Prefer a candidate with required metadata and no top-level symlinks.
- If every candidate is symlink-backed, keep the real artifact for vLLM loading but use the safe topology-fixture fallback in the next section.
- A large Hub download is disabled by default.


In [6]:
from pathlib import Path
import json
import os
import re
import shutil

from huggingface_hub import snapshot_download

print("Resolving Qwen TP=2 sharded_state artifact...")

# ---------------------------------------------------------------------
# 1. Prefer an explicitly supplied local path.
# ---------------------------------------------------------------------
explicit = os.environ.get("KAGGLE_VLLM_QWEN_PATH")
qwen_path = None
resolution_method = None

if explicit:
    candidate = Path(explicit).expanduser().resolve()
    if candidate.is_dir():
        qwen_path = candidate
        resolution_method = "KAGGLE_VLLM_QWEN_PATH"

# ---------------------------------------------------------------------
# 2. Look for an attached Kaggle Input artifact.
# ---------------------------------------------------------------------
if qwen_path is None:
    input_root = Path("/kaggle/input")

    if input_root.is_dir():
        matches = sorted(
            input_root.rglob("model-rank-0-part-0.safetensors")
        )

        if matches:
            # A valid TP=2 sharded_state directory should contain the
            # expected rank/part files plus config/tokenizer metadata.
            for match in matches:
                candidate = match.parent.resolve()

                expected = {
                    f"model-rank-{rank}-part-{part}.safetensors"
                    for rank in (0, 1)
                    for part in (0, 1)
                }

                names = {p.name for p in candidate.iterdir()}

                if (
                    expected.issubset(names)
                    and (candidate / "config.json").exists()
                    and (candidate / "tokenizer_config.json").exists()
                ):
                    qwen_path = candidate
                    resolution_method = "attached-kaggle-input"
                    break

# ---------------------------------------------------------------------
# 3. Look for a copy already present in /kaggle/working.
#    This also makes rerunning this notebook inexpensive.
# ---------------------------------------------------------------------
if qwen_path is None:
    working_root = Path("/kaggle/working")

    if working_root.is_dir():
        matches = sorted(
            working_root.rglob("model-rank-0-part-0.safetensors")
        )

        for match in matches:
            candidate = match.parent.resolve()

            # Do not accidentally choose the tiny topology test fixture.
            if "inspection-fixture" in str(candidate):
                continue
            if "symlink-test" in str(candidate):
                continue

            expected = {
                f"model-rank-{rank}-part-{part}.safetensors"
                for rank in (0, 1)
                for part in (0, 1)
            }

            names = {p.name for p in candidate.iterdir()}

            if (
                expected.issubset(names)
                and (candidate / "config.json").exists()
                and (candidate / "tokenizer_config.json").exists()
            ):
                qwen_path = candidate
                resolution_method = "existing-kaggle-working-copy"
                break

# ---------------------------------------------------------------------
# 4. If the artifact is not attached, download the EXISTING immutable
#    Hugging Face representation. This does NOT regenerate model shards.
# ---------------------------------------------------------------------
if qwen_path is None:
    print(
        "No attached/local Qwen TP=2 artifact was found.\n"
        "Downloading the existing published Hugging Face artifact..."
    )

    download_dir = WORK / "qwen2.5-3b-t4x2-sharded"

    download_dir.mkdir(parents=True, exist_ok=True)

    downloaded = snapshot_download(
        repo_id=MODEL_REPO,
        revision=MODEL_REVISION,
        local_dir=str(download_dir),
    )

    qwen_path = Path(downloaded).resolve()
    resolution_method = "huggingface-snapshot-download"

# ---------------------------------------------------------------------
# 5. Validate the resolved directory before continuing.
# ---------------------------------------------------------------------
assert qwen_path is not None
assert qwen_path.is_dir(), qwen_path

expected_shards = {
    f"model-rank-{rank}-part-{part}.safetensors"
    for rank in (0, 1)
    for part in (0, 1)
}

actual_shards = sorted(
    p.name
    for p in qwen_path.iterdir()
    if re.fullmatch(
        r"model-rank-\d+-part-\d+\.safetensors",
        p.name,
    )
)

assert set(actual_shards) == expected_shards, {
    "path": str(qwen_path),
    "found": actual_shards,
    "expected": sorted(expected_shards),
}

assert (qwen_path / "config.json").exists(), (
    f"config.json missing from {qwen_path}"
)

assert (qwen_path / "tokenizer_config.json").exists(), (
    f"tokenizer_config.json missing from {qwen_path}"
)

# ---------------------------------------------------------------------
# 6. Record actual shard characteristics.
# ---------------------------------------------------------------------
artifact_shard_records = []

for name in actual_shards:
    p = qwen_path / name

    artifact_shard_records.append(
        {
            "name": name,
            "is_symlink": p.is_symlink(),
            "resolved_path": str(p.resolve()),
            "size": p.stat().st_size,
        }
    )

chosen_summary = {
    "path": str(qwen_path),
    "resolution_method": resolution_method,
    "has_config": (qwen_path / "config.json").exists(),
    "has_tokenizer_config": (
        qwen_path / "tokenizer_config.json"
    ).exists(),
    "shards": actual_shards,
    "symlinks": sorted(
        p.name
        for p in qwen_path.iterdir()
        if p.is_symlink()
    ),
}

print("\nQwen artifact resolved successfully.")
print("Resolution method:", resolution_method)
print("Path:", qwen_path)
print("Expected TP=2 shards:", sorted(expected_shards))
print("\nArtifact summary:")
print(json.dumps(chosen_summary, indent=2))

print("\nShard records:")
print(json.dumps(artifact_shard_records, indent=2))

Resolving Qwen TP=2 sharded_state artifact...
No attached/local Qwen TP=2 artifact was found.


Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

SHARDED_STATE_SHA256SUMS.txt:   0%|          | 0.00/392 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]


Qwen artifact resolved successfully.
Resolution method: huggingface-snapshot-download
Path: /kaggle/working/kaggle-vllm-020-qwen-regression/qwen2.5-3b-t4x2-sharded
Expected TP=2 shards: ['model-rank-0-part-0.safetensors', 'model-rank-0-part-1.safetensors', 'model-rank-1-part-0.safetensors', 'model-rank-1-part-1.safetensors']

Artifact summary:
{
  "path": "/kaggle/working/kaggle-vllm-020-qwen-regression/qwen2.5-3b-t4x2-sharded",
  "resolution_method": "huggingface-snapshot-download",
  "has_config": true,
  "has_tokenizer_config": true,
  "shards": [
    "model-rank-0-part-0.safetensors",
    "model-rank-0-part-1.safetensors",
    "model-rank-1-part-0.safetensors",
    "model-rank-1-part-1.safetensors"
  ],
  "symlinks": []
}

Shard records:
[
  {
    "name": "model-rank-0-part-0.safetensors",
    "is_symlink": false,
    "resolved_path": "/kaggle/working/kaggle-vllm-020-qwen-regression/qwen2.5-3b-t4x2-sharded/model-rank-0-part-0.safetensors",
    "size": 2138586872
  },
  {
    "name

## Structural inspection, TP=2 identity, and topology mismatch rejection

`inspect_sharded_model()` deliberately rejects symlink-backed directories. That is a security property, not a model corruption signal.

When the attached artifact is symlink-free, inspect it directly.

When the mount contains symlinks, this notebook first **expects direct inspection to reject it**, then creates a tiny regular-file fixture that mirrors only:

- the four real shard names;
- required `config.json`;
- required `tokenizer_config.json`.

The fixture contains **zero-byte shard placeholders** and is used only for topology inspection. Actual Qwen loading later still uses the real model artifact.


In [7]:
from kaggle_vllm import inspect_sharded_model
from kaggle_vllm.exceptions import ShardedModelError

source_symlinks = chosen_summary["symlinks"]
inspection_mode = None
direct_symlink_rejection = False

if not source_symlinks:
    inspection = inspect_sharded_model(
        qwen_path,
        expected_tensor_parallel_size=2,
    )
    inspection_mode = "direct-real-artifact"
else:
    try:
        inspect_sharded_model(
            qwen_path,
            expected_tensor_parallel_size=2,
        )
    except ShardedModelError as exc:
        message = str(exc)
        assert "symlink" in message.casefold(), message
        direct_symlink_rejection = True
        print("Expected direct strict-inspection rejection:", message)
    else:
        raise AssertionError(
            "Symlink-backed source unexpectedly bypassed strict inspection policy"
        )

    if FIXTURE_ROOT.exists():
        shutil.rmtree(FIXTURE_ROOT)
    FIXTURE_ROOT.mkdir(parents=True)

    # Dereference and copy only the two small required metadata files.
    for metadata_name in ("config.json", "tokenizer_config.json"):
        source = qwen_path / metadata_name
        data = source.read_bytes()
        (FIXTURE_ROOT / metadata_name).write_bytes(data)

    # Topology-only placeholders: no model weights are copied.
    for shard_name in actual_shards:
        (FIXTURE_ROOT / shard_name).write_bytes(b"")

    inspection = inspect_sharded_model(
        FIXTURE_ROOT,
        expected_tensor_parallel_size=2,
    )
    inspection_mode = "topology-fixture-for-symlink-backed-artifact"

assert inspection.valid, inspection.to_dict()
assert inspection.rank_count == 2
assert {item.name for item in inspection.shards} == EXPECTED_SHARDS

inspection_root = qwen_path if inspection_mode == "direct-real-artifact" else FIXTURE_ROOT
wrong_topology = inspect_sharded_model(
    inspection_root,
    expected_tensor_parallel_size=1,
)
assert not wrong_topology.valid
assert any(
    "tensor parallel size 1" in item
    for item in wrong_topology.topology_errors
), wrong_topology.topology_errors

print("Inspection mode:", inspection_mode)
print(json.dumps(inspection.to_dict(), indent=2))
print("TP=2 topology and TP mismatch rejection: PASS")


Inspection mode: direct-real-artifact
{
  "path": "/kaggle/working/kaggle-vllm-020-qwen-regression/qwen2.5-3b-t4x2-sharded",
  "rank_count": 2,
  "total_size": 6172262512,
  "shards": [
    {
      "name": "model-rank-0-part-0.safetensors",
      "rank": 0,
      "part": 0,
      "size": 2138586872
    },
    {
      "name": "model-rank-0-part-1.safetensors",
      "rank": 0,
      "part": 1,
      "size": 947544384
    },
    {
      "name": "model-rank-1-part-0.safetensors",
      "rank": 1,
      "part": 0,
      "size": 2138586872
    },
    {
      "name": "model-rank-1-part-1.safetensors",
      "rank": 1,
      "part": 1,
      "size": 947544384
    }
  ],
  "metadata_files": [
    ".gitattributes",
    "LICENSE",
    "NOTICE",
    "README.md",
    "SHARDED_STATE_SHA256SUMS.txt",
    "config.json",
    "generation_config.json",
    "merges.txt",
    "model.safetensors.index.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "vocab.json"
  ],
  "missing_metadata": [],


## Explicit symlink-safety regression

This synthetic fixture verifies that `inspect_sharded_model()` continues to reject a symlinked shard. It does not modify the published Qwen artifact.


In [8]:
fixture = Path(
    tempfile.mkdtemp(
        prefix="kaggle-vllm-symlink-test-",
        dir="/kaggle/working",
    )
)

try:
    outside = fixture / "outside.safetensors"
    outside.write_bytes(b"not-a-model")

    model_fixture = fixture / "model"
    model_fixture.mkdir()

    (model_fixture / "config.json").write_text("{}", encoding="utf-8")
    (model_fixture / "tokenizer_config.json").write_text("{}", encoding="utf-8")
    (model_fixture / "model-rank-0-part-0.safetensors").symlink_to(outside)

    try:
        inspect_sharded_model(
            model_fixture,
            expected_tensor_parallel_size=1,
        )
    except ShardedModelError as exc:
        assert "symlink" in str(exc).casefold(), str(exc)
    else:
        raise AssertionError("symlinked shard was not rejected")
finally:
    shutil.rmtree(fixture)

print("Synthetic symlink rejection: PASS")


Synthetic symlink rejection: PASS


## Real Qwen TP=2 sharded-state load and short generation

The actual checkpoint is loaded in a dedicated child process. This avoids relying on private in-process vLLM shutdown internals and gives a stronger cleanup boundary: after the child exits, its TP worker processes must be gone.

This uses the **real** `qwen_path`, not the topology fixture.


In [9]:
qwen_smoke = r'''
from pathlib import Path
import json
import sys

from kaggle_vllm import KaggleLLM
from vllm import SamplingParams

model = sys.argv[1]

llm = KaggleLLM(
    model=model,
    load_format="sharded_state",
    tensor_parallel_size=2,
    dtype="float16",
    max_model_len=2048,
    gpu_memory_utilization=0.70,
    enforce_eager=True,
    disable_custom_all_reduce=True,
)

outputs = llm.generate(
    ["Explain tensor parallel inference in one concise sentence."],
    SamplingParams(
        temperature=0.0,
        max_tokens=48,
    ),
)

assert outputs
assert outputs[0].outputs
text = outputs[0].outputs[0].text
assert text.strip(), text

print(json.dumps({
    "status": "PASS",
    "model": model,
    "text": text,
}, ensure_ascii=False))
'''

QWEN_LOG.parent.mkdir(parents=True, exist_ok=True)
QWEN_LOG.write_text("QWEN TP=2 CHILD PROCESS LOG STARTED\n", encoding="utf-8")
print("Qwen child log created before launch:", QWEN_LOG)
with QWEN_LOG.open("a", encoding="utf-8") as log:
    proc = subprocess.run(
        [sys.executable, "-c", qwen_smoke, str(qwen_path)],
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True,
        env=os.environ.copy(),
        timeout=600,
    )

log_text = QWEN_LOG.read_text(encoding="utf-8", errors="replace")
print(log_text[-8000:])

assert proc.returncode == 0, (
    f"Qwen TP=2 child process failed with code {proc.returncode}. "
    f"See {QWEN_LOG}"
)

# subprocess.run() returning means the child and the TP workers it owned have
# passed the process boundary. Give CUDA/vLLM a short settle window and confirm
# no matching child command remains.
time.sleep(3)

ps = subprocess.run(
    ["ps", "-eo", "pid=,ppid=,args="],
    check=True,
    capture_output=True,
    text=True,
).stdout

survivors = [
    line.strip()
    for line in ps.splitlines()
    if str(qwen_path) in line
    and "python" in line.casefold()
    and str(os.getpid()) not in line
]

assert not survivors, f"Qwen/vLLM processes survived child exit: {survivors}"

print("Real Qwen TP=2 sharded_state load, generation, and process cleanup: PASS")


Qwen child log created before launch: /kaggle/working/kaggle-vllm-020-qwen-regression/qwen-tp2-generation.log
sions: norm_quant, act_quant
WARNING 08-31 12:36:33 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore pid=280) INFO 08-31 12:36:50 [core.py:103] Initializing a V1 LLM engine (v0.18.2.dev0+ga26e8dc7f.d20260822) with config: model='/kaggle/working/kaggle-vllm-020-qwen-regression/qwen2.5-3b-t4x2-sharded', speculative_config=None, tokenizer='/kaggle/working/kaggle-vllm-020-qwen-regression/qwen2.5-3b-t4x2-sharded', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=sharded_state, tensor_parallel_size=2, pipeline_parallel_size=1, da

## Machine-readable result

A PASS here means:

- public `kaggle-vllm==0.2.0` was used;
- exact immutable native runtime identity was verified;
- Kaggle Torch remained unchanged;
- the real Qwen artifact exposes the expected four TP=2 rank/part shard files;
- direct strict inspection passed **or** a symlink-backed mount was correctly rejected and topology was validated through a non-weight fixture;
- topology mismatch and explicit symlink rejection worked;
- the real Qwen artifact loaded with vLLM `sharded_state` at TP=2;
- short generation succeeded;
- the child-process runtime exited cleanly.

The evidence records the inspection mode so a fixture-based topology check is never misrepresented as direct inspection of symlink-backed files.


In [10]:
result = {
    "status": "PASS",
    "sdk_version": kaggle_vllm.__version__,
    "model_repository": MODEL_REPO,
    "model_revision": MODEL_REVISION,
    "model_path": str(qwen_path),
    "native_wheel": manifest["wheel"],
    "torch_before": torch_before,
    "torch_after": torch_after,
    "artifact_shards": artifact_shard_records,
    "source_top_level_symlinks": source_symlinks,
    "inspection_mode": inspection_mode,
    "direct_symlink_rejection": direct_symlink_rejection,
    "inspection": inspection.to_dict(),
    "qwen_log": str(QWEN_LOG),
    "checks": {
        "public_sdk_0_2_0": True,
        "strict_bootstrap": True,
        "native_identity": True,
        "strict_doctor": True,
        "torch_preserved": True,
        "real_artifact_four_shards": True,
        "tp2_topology": True,
        "topology_mismatch_rejection": True,
        "symlink_policy": True,
        "real_tp2_sharded_state_load": True,
        "short_generation": True,
        "clean_child_process_exit": True,
    },
}

EVIDENCE.write_text(
    json.dumps(result, indent=2) + "\n",
    encoding="utf-8",
)

print(json.dumps(result, indent=2))
SENTINEL.write_text(json.dumps({
    "identity": IDENTITY,
    "status": "PASS",
    "evidence": str(EVIDENCE),
    "qwen_log": str(QWEN_LOG),
}, indent=2) + "\n", encoding="utf-8")
print("Evidence:", EVIDENCE)
print("FINAL QWEN TP=2 REGRESSION: PASS")


{
  "status": "PASS",
  "sdk_version": "0.2.0",
  "model_repository": "waqasm86/kaggle-vllm-models",
  "model_revision": "08bb62d0b68d20062e9009a9769c0df53d3dae21",
  "model_path": "/kaggle/working/kaggle-vllm-020-qwen-regression/qwen2.5-3b-t4x2-sharded",
  "native_wheel": {
    "filename": "vllm-0.18.2.dev0+ga26e8dc7f.d20260822.cu128-cp312-cp312-linux_x86_64.whl",
    "sha256": "5a9bd710b8a19fdd23abb3442baad892da977466f996334decd533a225f5fd0c",
    "hf_repo_id": "waqasm86/kaggle-vllm-binaries",
    "hf_revision": "f6b4f10de54924ed6fe9e28cceab84eca7276ab6",
    "resolved_path": "/kaggle/working/kaggle-vllm-cache/vllm-0.18.2.dev0+ga26e8dc7f.d20260822.cu128-cp312-cp312-linux_x86_64.whl"
  },
  "torch_before": {
    "version": "2.10.0+cu128",
    "cuda": "12.8",
    "path": "/usr/local/lib/python3.12/dist-packages/torch/__init__.py"
  },
  "torch_after": {
    "version": "2.10.0+cu128",
    "cuda": "12.8",
    "path": "/usr/local/lib/python3.12/dist-packages/torch/__init__.py"
  },
  "art